# Generative AI Optimized Recommendations with Custom Dataset (boto3)

This notebook runs a **SageMaker AI Recommendation Job** using a custom dataset from S3, with model optimization enabled. The service evaluates your model against the provided dataset, applies optimizations like speculative decoding, and returns a `ModelPackage` with instance-specific `InferenceSpecifications` and predicted performance metrics.

**Workflow:**
1. (Optional) Download model weights and upload to S3
2. Create a workload config referencing a custom dataset from S3
3. Submit a recommendation job with `OptimizeModel=True` to enable deep optimizations
4. Poll until completion
5. Review the recommended configuration and expected performance
6. Clean up resources

**Prerequisites:**
- Model weights in S3 in HuggingFace SafeTensor format
- Custom dataset (e.g. `gsm8k_training.jsonl`) uploaded to an S3 folder
- An IAM execution role with SageMaker, S3, ECR, and CloudWatch Logs permissions
- Caller IAM identity with `sagemaker:*` permissions
- Reserved capacity (Training Plan ARN) if targeting `ml.p5en.48xlarge`

## Step 1 — Install dependencies

Upgrade boto3 to ensure the latest API features are available.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

## Step 2 — Configuration

**You must provide your own values below before running this notebook:**
- `MODEL_S3_URI` — S3 path to your model weights in HuggingFace SafeTensor format (e.g. `s3://your-bucket/models/gpt-oss-20b/`)
- `S3_OUTPUT` — S3 path where recommendation outputs will be written (e.g. `s3://your-bucket/recommendations/`)
- `DATASET_S3_URI` — S3 folder containing your custom dataset file (must end with `/`)

The example below uses `openai/gpt-oss-20b` as the model. If you don't have it in S3 yet, see the **optional download cell** in Step 2b.

> **Note:** `ROLE_ARN` is automatically retrieved from the SageMaker execution role attached to this Studio environment via `sagemaker.get_execution_role()`.

In [ ]:
import json
import time
import boto3
from datetime import datetime
import uuid

def get_execution_role():
    sts = boto3.client('sts')
    arn = sts.get_caller_identity()['Arn']
    if ':assumed-role/' in arn:
        parts = arn.split(':')
        account = parts[4]
        role_name = parts[5].split('/')[1]
        return f"arn:aws:iam::{account}:role/{role_name}"
    raise RuntimeError(f"Cannot derive execution role from ARN: {arn}. Set ROLE_ARN manually.")

REGION          = "us-west-2"
MODEL_S3_URI    = "s3://your-bucket/models/gpt-oss-20b/"       # <-- replace with your S3 path
S3_OUTPUT       = "s3://your-bucket/recommendations/"          # <-- replace with your S3 output path
DATASET_S3_URI  = "s3://your-bucket/datasets/"                 # <-- replace with your dataset S3 folder (must end with /)

# Auto-generate unique names on every run
run_id      = datetime.now().strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
CONFIG_NAME = f"rec-config-{run_id}"
JOB_NAME    = f"rec-job-{run_id}"

session = boto3.Session(region_name=REGION)
sm = session.client("sagemaker")
ROLE_ARN = get_execution_role()

print(f"Role ARN    : {ROLE_ARN}")
print(f"Config name : {CONFIG_NAME}")
print(f"Job name    : {JOB_NAME}")

### Ensure role trust policy allows SageMaker

Checks the trust policy of the execution role and adds `sagemaker.amazonaws.com` if missing. This is a no-op if the trust policy is already correct.

In [ ]:
iam = boto3.client('iam')
role_name = ROLE_ARN.split('/')[-1]

trust = iam.get_role(RoleName=role_name)['Role']['AssumeRolePolicyDocument']
principals = [
    s.get('Principal', {}).get('Service', '')
    for s in trust.get('Statement', [])
]
flat = [p for item in principals for p in (item if isinstance(item, list) else [item])]

if 'sagemaker.amazonaws.com' in flat:
    print('✅ Trust policy already allows sagemaker.amazonaws.com')
else:
    trust['Statement'].append({
        'Effect': 'Allow',
        'Principal': {'Service': 'sagemaker.amazonaws.com'},
        'Action': 'sts:AssumeRole'
    })
    iam.update_assume_role_policy(
        RoleName=role_name,
        PolicyDocument=json.dumps(trust)
    )
    print(f'✅ Added sagemaker.amazonaws.com to trust policy for role: {role_name}')

## Step 2b — (Optional) Download openai/gpt-oss-20b and upload to S3

Skip this step if your model weights are already in S3.

This cell downloads `openai/gpt-oss-20b` from Hugging Face using `hf_transfer` for fast multi-part download, then syncs the weights to your S3 bucket. Update `MODEL_S3_URI` in Step 2 to match your bucket before running.

In [ ]:
# Optional: download gpt-oss-20b from Hugging Face and upload to S3
# Uncomment and run this cell if you need to upload the model to S3 first.
#
# LOCAL_MODEL_DIR = "/tmp/gpt-oss-20b"
#
# !pip install -q "huggingface_hub[hf_transfer]"
# !HF_HUB_ENABLE_HF_TRANSFER=1 huggingface-cli download openai/gpt-oss-20b \
#     --local-dir {LOCAL_MODEL_DIR} \
#     --exclude '*.gguf' '*.bin'
#
# !aws s3 sync {LOCAL_MODEL_DIR} {MODEL_S3_URI}
# print(f"Model uploaded to {MODEL_S3_URI}")

In [ ]:
# Uncomment and run this cell if you need to upload the model to S3 first.
#
# !pip install -q "huggingface_hub[hf_transfer]"
# !HF_HUB_ENABLE_HF_TRANSFER=1 huggingface-cli download openai/gpt-oss-20b \
#     --local-dir /tmp/gpt-oss-20b \
#     --exclude '*.gguf' '*.bin'
#
# !aws s3 sync /tmp/gpt-oss-20b {MODEL_S3_URI}
# print(f"Model uploaded to {MODEL_S3_URI}")

## Step 3 — Create workload config with custom dataset

This workload config uses a **custom dataset** (`gsm8k_training.jsonl`) from S3 to represent real traffic patterns.

**Key parameters:**
| Parameter | Value | Description |
|---|---|---|
| `custom_dataset_type` | `generic` | Use a custom JSONL dataset |
| `input_file` | `/opt/ml/input/data/datasets/...` | Path to the dataset inside the container, mapped from the S3 `datasets` channel |
| `concurrency` | `64` | Simulates 64 concurrent users |
| `benchmark_duration` | `300` | Runs for 5 minutes after a 30-second warmup |
| `prompt_input_tokens_mean` | `3500` | Long-context workload |
| `extra_inputs` | `ignore_eos:true temperature:0` | Forces deterministic, full-length outputs |

> **S3 URI note:** The `S3Uri` in `DatasetConfig` must always end with `/`. SageMaker downloads everything under that prefix to `/opt/ml/input/data/<ChannelName>/`. Set `input_file` to the specific file within that mount point.

In [ ]:
# 1. Create workload config
workload_spec = {
    "benchmark": {"type": "aiperf"},
    "parameters": {
        "custom_dataset_type": "generic",
        "input_file": "/opt/ml/input/data/datasets/gsm8k_training.jsonl",
        "concurrency": 64,
        "benchmark_duration": 300,
        "streaming": True,
        "warmup_duration": 30,
        "output_tokens_mean": 200,
        "output_tokens_stddev": 0,
        "extra_inputs": "ignore_eos:true temperature:0",
        "prompt_input_tokens_mean": 3500,
        "random_seed": 42,
    },
}

config_resp = sm.create_ai_workload_config(
    AIWorkloadConfigName=CONFIG_NAME,
    DatasetConfig={
        "InputDataConfig": [
            {
                "ChannelName": "datasets",
                "DataSource": {
                    "S3DataSource": {
                        # Must be a folder path ending with /, not a file path
                        "S3Uri": DATASET_S3_URI
                    }
                }
            }
        ]
    },
    AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(workload_spec)}},
)
print(f"Config created: {config_resp['AIWorkloadConfigArn']}")

## Step 4 — Submit recommendation job

Launches the recommendation job with model optimization enabled. With `OptimizeModel=True`, the service will evaluate and apply deep optimizations such as speculative decoding (EAGLE 3), quantization, and kernel tuning.

**Key parameters:**
| Parameter | Value | Description |
|---|---|---|
| `PerformanceTarget` | `throughput` | Optimize for maximum output token throughput |
| `ComputeSpec.InstanceTypes` | `ml.p5en.48xlarge` | Target instance (8×H100 SXM5 80GB) |
| `CapacityReservationConfig` | Training Plan ARNs | Use reserved capacity to guarantee instance availability |
| `OptimizeModel` | `True` | Enable speculative decoding, quantization, and kernel tuning |
| `InferenceSpecification.Framework` | `LMI` | Use the DJL Large Model Inference container |

The output is a `ModelPackage` with an `InferenceSpecification` per recommended instance. Each spec includes the container image, environment variables, and `ExpectedPerformance` metrics.

In [ ]:
# 2. Create recommendation job
job_resp = sm.create_ai_recommendation_job(
    AIRecommendationJobName=JOB_NAME,
    ModelSource={
        "S3": {
            "S3Uri": MODEL_S3_URI
        }
    },
    OutputConfig={"S3OutputLocation": S3_OUTPUT},
    AIWorkloadConfigIdentifier=CONFIG_NAME,
    RoleArn=ROLE_ARN,
    PerformanceTarget={
        "Constraints": [
            {"Metric": "throughput"},
        ]
    },
    ComputeSpec={
        "InstanceTypes": ["ml.p5en.48xlarge"],
        "CapacityReservationConfig": {
            "CapacityReservationPreference": "capacity-reservations-only",
            "MlReservationArns": [
                "arn:aws:sagemaker:us-west-2:276831242074:training-plan/training-plan-1",
                "arn:aws:sagemaker:us-west-2:276831242074:training-plan/training-plan"
            ]
        }
    },
    OptimizeModel=True,
    InferenceSpecification = {"Framework": "LMI"},
)
print(f"Job created: {job_resp['AIRecommendationJobArn']}")

## Step 5 — Poll for completion

Polls every 30 seconds until the job reaches a terminal state: `Completed`, `Failed`, or `Stopped`.

In [ ]:
# 3. Poll until terminal
while True:
    desc = sm.describe_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)
    status = desc["AIRecommendationJobStatus"]
    print(f"  Status: {status}")
    if status in ("Completed", "Failed", "Stopped"):
        print(desc)
        break
    time.sleep(30)

## Step 6 — Review recommendations

Each recommendation includes:
- **`OptimizationDetails`** — optimizations applied (e.g. `SpeculativeDecoding` with EAGLE 3, `Quantization`)
- **`ModelDetails`** — `ModelPackageArn` and `InferenceSpecificationName` to use when deploying via `CreateModel`
- **`DeploymentConfiguration`** — container image URI, instance type, copy count, and environment variables (e.g. `SM_VLLM_SPECULATIVE_CONFIG`, `SM_VLLM_TENSOR_PARALLEL_SIZE`)
- **`ExpectedPerformance`** — predicted `RequestLatency` (p50/p90/p99), `OutputTokenThroughput`, `TimeToFirstToken`, and `InterTokenLatency`

In [ ]:
# 4. Print recommendations
if status == "Completed":
    recommendations = desc.get("Recommendations", [])
    print(f"\n{len(recommendations)} recommendation(s) found:")
    for i, rec in enumerate(recommendations, 1):
        print(f"\n--- Recommendation {i} ---")
        print(f"  {rec.get('RecommendationDescription', 'N/A')}")
        if rec.get("InstanceDetails"):
            for inst in rec["InstanceDetails"]:
                print(f"  Instance: {inst['InstanceType']} x{inst.get('InstanceCount', 1)}")
        if rec.get("OptimizationDetails"):
            for opt in rec["OptimizationDetails"]:
                print(f"  Optimization: {opt['OptimizationType']}")
        if rec.get("ExpectedPerformance"):
            for perf in rec["ExpectedPerformance"]:
                print(f"  {perf['Metric']}: {perf['Value']} {perf.get('Unit', '')}")
        if rec.get("DeploymentConfiguration"):
            deploy = rec["DeploymentConfiguration"]
            print(f"  Deploy image: {deploy.get('ImageUri', 'N/A')}")
            print(f"  Deploy instance: {deploy.get('InstanceType', 'N/A')}")
else:
    print(f"\nEnded with status: {status}")
    if "FailureReason" in desc:
        print(f"Reason: {desc['FailureReason']}")

In [ ]:
print(json.dumps(desc.get("Recommendations", []), indent=2))

## Step 7 — Cleanup

Delete the recommendation job and workload config to avoid leaving unused resources in your account.

> **Note:** A job must be in a terminal state before it can be deleted. Call `stop_ai_recommendation_job` first if the job is still running.

In [ ]:
sm.stop_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)

In [ ]:
sm.delete_ai_recommendation_job(AIRecommendationJobName=JOB_NAME)

In [ ]:
sm.delete_ai_workload_config(AIWorkloadConfigName=CONFIG_NAME)